# 06.13_read_scBasalMetazoans_h5ad_Python

读取整合 scBasalMetazoans 图谱与绘图。

- 当前文件：`analysis/06_single_cell_analysis/06.13_read_scBasalMetazoans_h5ad_Python.ipynb`
- 原始来源：`Codes/06.11_read_scBasalMetazoans_h5ad.ipynb`（旧编号仅用于溯源）。
- 运行内核：**python**。
- 导入依赖：`json`, `numpy`, `pandas`, `scanpy`。
- 当前编号与流程见 `docs/workflow.md`、`docs/code_index.md`。
- 仅更新整理版导读；原分析单元格、参数和顺序保持不变。原始 cell 索引在本文件中加 1。


In [ ]:
import scanpy as sc

In [ ]:
# 读取sc_BasalMetazoa数据集（MD5：30917bb0afde23f7138c2b7c681249f8  sc_BasalMetazoa.normalized.gzip.h5ad）
# 数据介绍：
# 涉及4个后生动物门类，共9个物种整合后的单细胞数据集
# 按照进化时间顺序排列为：多孔动物（海绵Spla），扁盘动物（4种丝盘虫：ClH23、HoH13、TrH2、TrH1），刺胞动物（海月水母Auco，半球美螅水母Clhe，海葵Neve），两侧动物（斑马鱼Dare）
# 整合后数据集共173,786细胞 × 2,216同源基因（Orthogroup）
adata = sc.read_h5ad('/share/home/zhangze/zz/NeuralOrigin/Data/06.SingleCellAnalysis/SingleCellIntegrated/Seurat_RPCA_to_Scanpy/sc_BasalMetazoa.normalized.gzip.h5ad')
adata

In [ ]:
# CellTypes：每个物种单独的原始细胞类型标签
# 格式为“物种名_原始细胞类型”，比如“Auco_NE, Neural cell”代表海月水母神经细胞
sc.pl.umap(adata, color=['CellTypes'])

In [ ]:
# Phylum：门类标签
# species：物种标签
# BroadType：整合后重注释细胞类型标签
sc.pl.umap(adata, color=['Phylum', 'species', 'BroadType'])

In [ ]:
import json
import numpy as np
import pandas as pd

def export_split_atlases(adata):
    # 基础输出路径
    base_path = "/share/home/zhangze/zz/NeuralOrigin/Figures/"
    
    # --- 配置 1: 物种颜色映射 ---
    species_color_map = {
        "Spla":"#fba414", "ClH23":"#ffa8a7", "HoH13":"#eb7f7f", 
        "TrH2":"#ff5d4e", "TrH1":"#EC2B24", "Auco":"#2A52BE", 
        "Clhe":"#4374B3", "Neve":"#6DA0E2", "Dare":"#43b244"
    }
    
    # --- 配置 2: 细胞类型颜色映射 ---
    broad_color_map = {
        'Cnidocytes': '#17becf',
        'Epidermal/Muscle': '#9467bd',
        'Gland': '#d62728',
        'Sensory': '#8c564b',
        'Neural': '#2ca02c',
        'Stem/Germline': '#fedb61',
        'Unknow': '#707070'
    }

    # 获取 UMAP 坐标
    coords = adata.obsm['X_umap'].astype(np.float32)

    # --- 任务 A: 导出物种分布 JSON ---
    sp_categories = list(species_color_map.keys())
    sp_to_idx = {sp: i for i, sp in enumerate(sp_categories)}
    sp_indices = adata.obs['species'].map(lambda x: sp_to_idx.get(x, -1)).values
    
    # 结构: [[x, y, sp_idx], ...]
    sp_data = np.column_stack([coords[:, 0], coords[:, 1], sp_indices]).tolist()
    
    with open(base_path + "Integrated_umap_species.json", "w") as f:
        json.dump({
            "categories": sp_categories,
            "palette": [species_color_map[s] for s in sp_categories],
            "data": sp_data
        }, f, separators=(',', ':'))

    # --- 任务 B: 导出细胞类型 JSON ---
    br_categories = list(broad_color_map.keys())
    br_to_idx = {br: i for i, br in enumerate(br_categories)}
    br_indices = adata.obs['BroadType'].map(lambda x: br_to_idx.get(x, -1)).values
    
    # 结构: [[x, y, br_idx], ...]
    br_data = np.column_stack([coords[:, 0], coords[:, 1], br_indices]).tolist()
    
    with open(base_path + "Integrated_umap_broadtype.json", "w") as f:
        json.dump({
            "categories": br_categories,
            "palette": [broad_color_map[b] for b in br_categories],
            "data": br_data
        }, f, separators=(',', ':'))

    print(f"成功导出两个文件至 {base_path}")

# 执行
export_split_atlases(adata)

In [ ]:
# 提取 BroadType 为 Neural 的所有细胞
adata_NE = adata[adata.obs['BroadType'] == 'Neural'].copy()
adata_NE

In [ ]:
sc.pl.umap(adata_NE, color=['Phylum', 'species'])